# OASIS INFOBYTE Internship Project
## Sentiment Analysis Pipeline

**Organization:** OASIS INFOBYTE  
**Track:** Data Analytics  
**Task:** Level 1 - Task 4:  Sentiment Analysis  
**Author:** Ahmed Mahmoud Khalifa  

---


# Oasis Infobyte Data Science Internship - Task 4
## 📊 Professional End-to-End Sentiment Analysis Pipeline

### 🎯 Project Overview
This project builds a robust Natural Language Processing (NLP) pipeline to analyze and classify sentiments from social media posts and tweets.

### 🌟 Key Challenges & Solutions:
1. **High Granularity Labels (191 Classes)**: The raw dataset contains 191 unique emotion labels with trailing whitespaces and extreme sparsity (many classes with only 1–3 examples). We engineer a domain-aware mapping consolidating them into standard **Positive**, **Negative**, and **Neutral** categories.
2. **Severe Class Imbalance**: Handled via Stratified K-Fold validation and dynamic **Balanced Class Weighting** in loss functions.
3. **Social Media Noise**: Handled with tailored preprocessing (emoji demojizing, contraction expansion, URL/mention cleaning, sentiment-preserving stopword filtering, and lemmatization).
4. **Production Architecture**: Decoupled into reusable modules (`preprocess.py`, `label_engineering.py`, `train.py`, `predict.py`).

In [1]:
# 1. Environment Setup & Imports
import os
import sys
import re
import string
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Scikit-Learn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

# Configure visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

print("All libraries imported successfully!")

## 1. Data Loading & Initial Inspection

In [2]:
data_path = '../data/3) Sentiment dataset.csv' if os.path.exists('../data/3) Sentiment dataset.csv') else 'data/3) Sentiment dataset.csv'
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
display(df.head(5))

In [3]:
# Dataset Schema & Missing Value Check
print("--- Dataset Info ---")
df.info()
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

## 2. Exploratory Data Analysis (EDA)
### Investigating the 191 Granular Sentiment Classes

In [4]:
# Strip whitespace and count unique raw classes
df['Stripped_Sentiment'] = df['Sentiment'].astype(str).str.strip()
raw_class_count = df['Stripped_Sentiment'].nunique()
print(f"Raw unique classes after stripping whitespace: {raw_class_count}")

# Top 20 raw sentiment categories
plt.figure(figsize=(12, 6))
top20 = df['Stripped_Sentiment'].value_counts().head(20)
sns.barplot(x=top20.values, y=top20.index, palette='crest')
plt.title('Top 20 Most Frequent Raw Emotion Categories (out of 191)', fontsize=14, fontweight='bold')
plt.xlabel('Frequency')
plt.ylabel('Emotion Label')
plt.tight_layout()
plt.show()

In [5]:
# Distribution across Platforms and Engagement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Platform distribution
platform_counts = df['Platform'].str.strip().value_counts()
axes[0].pie(platform_counts.values, labels=platform_counts.index, autopct='%1.1f%%', colors=['#4267B2', '#E1306C', '#1DA1F2', '#333333'])
axes[0].set_title('Posts Share by Social Media Platform', fontweight='bold')

# Text character length
df['Char_Length'] = df['Text'].astype(str).str.len()
sns.histplot(df['Char_Length'], bins=30, kde=True, ax=axes[1], color='#3498db')
axes[1].set_title('Distribution of Post Character Lengths', fontweight='bold')
axes[1].set_xlabel('Character Count')

plt.tight_layout()
plt.show()

## 3. Label Engineering & Resolving Class Imbalance
We map the 191 granular emotions into a canonical 3-class taxonomy: `Positive`, `Negative`, `Neutral`.

In [6]:
from src.label_engineering import engineer_labels

df, meta = engineer_labels(df, sentiment_column='Sentiment')
print("Mapping Metadata:")
print(json.dumps(meta, indent=2))

# Visualizing Consolidated 3-Class Sentiment Distribution
plt.figure(figsize=(7, 5))
palette = {'Positive': '#2ecc71', 'Negative': '#e74c3c', 'Neutral': '#3498db'}
counts = df['Clean_Sentiment'].value_counts()
ax = sns.barplot(x=counts.index, y=counts.values, palette=palette, hue=counts.index, legend=False)
plt.title('Cleaned Sentiment Distribution (3 Standard Classes)', fontsize=14, fontweight='bold')
plt.ylabel('Number of Samples')
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{int(height)} ({height/len(df):.1%})',
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom', fontsize=11, fontweight='bold', xytext=(0, 4), textcoords='offset points')
plt.tight_layout()
plt.show()

## 4. NLP Preprocessing Pipeline
- Emoji translation to descriptive English (`😊` -> `smile face`)
- Expansion of contractions (`can't` -> `cannot`)
- Removal of URLs, user tags (`@`), and punctuation
- Lemmatization & sentiment-preserving stopword filtering

In [7]:
from src.preprocess import clean_text

# Demonstration on sample text
sample = "Enjoying a beautiful day at the park! 😊 @User123 can't wait for #Vacation https://example.com"
print("Original: ", sample)
print("Cleaned  : ", clean_text(sample))

# Apply preprocessing across entire dataset
df['Clean_Text'] = df['Text'].astype(str).apply(clean_text)
df[['Text', 'Clean_Text', 'Clean_Sentiment']].head(5)

In [8]:
# Word Clouds for Positive, Negative, and Neutral Sentiments
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
cmaps = {'Positive': 'Greens', 'Negative': 'Reds', 'Neutral': 'Blues'}

for idx, (cls, cmap) in enumerate(cmaps.items()):
    text_subset = df[df['Clean_Sentiment'] == cls]['Clean_Text'].dropna().str.cat(sep=' ')
    wc = WordCloud(width=600, height=400, background_color='white', colormap=cmap, max_words=100, random_state=42).generate(text_subset)
    axes[idx].imshow(wc, interpolation='bilinear')
    axes[idx].set_title(f'Word Cloud: {cls}', fontsize=14, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 5. Model Training & Cross-Validation Comparison
We benchmark 4 distinct algorithms using Stratified 5-Fold Cross-Validation:
1. Logistic Regression (with Balanced Class Weights)
2. Linear Support Vector Classifier (Calibrated for probability calibration)
3. Multinomial Naive Bayes
4. Random Forest Classifier (Balanced)

In [9]:
# Stratified Train-Test Split (80% Train, 20% Holdout Test)
X = np.array(df['Clean_Text'].astype(str).tolist())
y = np.array(df['Clean_Sentiment'].astype(str).tolist())
classes = sorted(list(np.unique(y)))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)}")

In [10]:
candidate_models = {
    'Logistic Regression (Balanced)': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Linear SVC (Calibrated)': CalibratedClassifierCV(LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)),
    'Multinomial Naive Bayes': MultinomialNB(),
    'Random Forest (Balanced)': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
}

cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
comparison_results = []

for name, clf in candidate_models.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True)),
        ('clf', clf)
    ])
    
    # 5-Fold Stratified CV
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv_folds, scoring='f1_weighted')
    
    # Fit and test on holdout
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    test_acc = accuracy_score(y_test, y_pred)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)
    
    comparison_results.append({
        'Model': name,
        'CV_F1_Weighted': round(float(np.mean(cv_scores)), 4),
        'CV_F1_Std': round(float(np.std(cv_scores)), 4),
        'Test_Accuracy': round(test_acc, 4),
        'Test_F1_Weighted': round(float(f1_w), 4)
    })

comp_df = pd.DataFrame(comparison_results).sort_values(by='CV_F1_Weighted', ascending=False).reset_index(drop=True)
display(comp_df)

## 6. Hyperparameter Tuning with GridSearchCV

In [11]:
base_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True)),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42))
])

param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_features': [3000, 5000, None],
    'tfidf__min_df': [1, 2],
    'clf__C': [0.1, 1.0, 5.0, 10.0],
    'clf__solver': ['lbfgs']
}

grid_search = GridSearchCV(base_pipeline, param_grid, cv=cv_folds, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print(f"Best CV F1-Weighted: {grid_search.best_score_:.4f}")
best_model = grid_search.best_estimator_

## 7. Final Model Evaluation on Holdout Test Set

In [12]:
y_test_pred = best_model.predict(X_test)
print("=== Classification Report (Test Set) ===\n")
print(classification_report(y_test, y_test_pred, labels=classes, zero_division=0))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_test_pred, labels=classes)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, annot_kws={'size': 14, 'weight': 'bold'})
plt.title('Confusion Matrix - Tuned Sentiment Pipeline', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Sentiment')
plt.ylabel('Actual Sentiment')
plt.tight_layout()
plt.show()

## 8. Interactive Real-World Prediction Test
Testing the trained pipeline on new unseen tweets.

In [13]:
from src.predict import SentimentPredictor

predictor = SentimentPredictor(model_path='../outputs/models/best_sentiment_model.joblib' if os.path.exists('../outputs/models/best_sentiment_model.joblib') else 'outputs/models/best_sentiment_model.joblib')

test_samples = [
    "Just finished an awesome workout! Feeling energetic and healthy! 💪🔥",
    "Traffic was completely awful today. Waited 2 hours in the rain. 😡",
    "Reading a research paper on natural language processing techniques.",
    "Can't believe how bad this customer support was! Total waste of time.",
    "A cozy evening with hot tea and a great book. Pure peace! 🍵📖"
]

for sample in test_samples:
    res = predictor.predict_single(sample)
    print(f"Post      : {res['original_text']}")
    print(f"Sentiment : {res['badge']} {res['predicted_sentiment']} (Confidence: {res['confidence']*100:.1f}%)")
    print(f"Probabilities: {res['probabilities']}")
    print("-" * 70)